# Building a RAG Application with Gemini

**Homework** — extends the three workshop notebooks
(`01_llm_architecture_and_embeddings`, `02_rag_and_architecture_patterns`,
`03_real_llm_rag_agentic_patterns`).

### Use case
A **study assistant for LLM / Transformer / RAG fundamentals**. It answers
questions grounded in a small collection of public Wikipedia articles and
refuses (says what is missing) when the articles do not contain the answer.

### What changes compared with the workshop

| Workshop notebook 03 (toy version) | This homework |
|---|---|
| Bag-of-words embedding written by hand | `models/gemini-embedding-001` |
| Python `list` used as the "index" | **Chroma** local vector database (persisted to disk) |
| Rule-based extractive answerer | **Gemini Flash** chat model |
| Prompt built by string concatenation | **LangChain** LCEL chain |
| No agent | **RAG agent** (LangGraph ReAct) + chain-vs-agent comparison |
| 5 synthetic in-notebook snippets | >= 3 real public documents loaded with metadata |

> Note: the `03_*.ipynb` file shipped with the workshop is the *transparent /
> local* version (no external APIs). This notebook builds the professional
> version described in the assignment (Gemini + LangChain + Chroma) on top of
> the same architecture.


## 0. Setup

1. Create a Gemini API key (free tier): https://aistudio.google.com/apikey
2. From the repository root, copy the env template and fill in the key:
   ```bash
   cp .env.example .env      # Windows: copy .env.example .env
   ```
   then edit `.env` and set `GOOGLE_API_KEY=...`
3. Create a virtual environment and install dependencies:
   ```bash
   python -m venv .venv
   .venv\Scripts\activate        # Windows
   # source .venv/bin/activate    # macOS / Linux
   pip install -r requirements.txt
   ```
4. Run this notebook top to bottom.

The `.env` file is git-ignored. Never commit real credentials.


In [ ]:
# Run once if you did not install from the terminal (uncomment):
# %pip install -r ../requirements.txt

In [ ]:
import os
import time
import datetime as dt

from dotenv import load_dotenv, find_dotenv

# find_dotenv walks up the tree, so this works whether the notebook is run
# from notebooks/ or from the repo root.
load_dotenv(find_dotenv())

# WebBaseLoader wants a User-Agent; set a default before importing loaders.
os.environ.setdefault("USER_AGENT", "rag-homework/1.0 (educational use)")

assert os.environ.get("GOOGLE_API_KEY"), (
    "GOOGLE_API_KEY not found. Copy .env.example to .env and add your Gemini key."
)

# ---- Configuration (recorded in the README) -------------------------------
CHAT_MODEL      = "gemini-3.6-flash"          # Gemini Flash, Developer API free tier
EMBEDDING_MODEL = "models/gemini-embedding-001"   # required by the assignment
TOP_K           = 4                            # chunks retrieved per query
CHUNK_SIZE      = 1200
CHUNK_OVERLAP   = 200
MAX_CHARS_PER_DOC = 24000                       # cap long articles (see markdown below)
PERSIST_DIR     = "../data/chroma"
COLLECTION_NAME = "llm_rag_wikipedia"

print("chat model     :", CHAT_MODEL)
print("embedding model:", EMBEDDING_MODEL)
print("retrieval top-k:", TOP_K)

## 1. Select and load the sources

### Sources (all public, CC BY-SA 4.0)

| id | Title | URL |
|----|-------|-----|
| S1 | Large language model | https://en.wikipedia.org/wiki/Large_language_model |
| S2 | Transformer (deep learning architecture) | https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture) |
| S3 | Retrieval-augmented generation | https://en.wikipedia.org/wiki/Retrieval-augmented_generation |
| S4 | Word embedding | https://en.wikipedia.org/wiki/Word_embedding |
| S5 | Vector database | https://en.wikipedia.org/wiki/Vector_database |

### Why these sources
They cover the full pipeline taught in the workshop: what an LLM is (S1), the
architecture that powers it (S2), how embeddings represent meaning (S4), how a
vector database stores and searches them (S5), and how retrieval is combined
with generation (S3). Together they let the assistant answer conceptual
questions about the same material as the workshop.

### Questions the application is expected to answer
- Definitional: *"What is self-attention?"*, *"What is a vector database?"*
- Explanatory: *"What problem does RAG solve?"*, *"Why do transformers need positional encoding?"*
- Comparative (partial evidence): *"Is RAG better than fine-tuning for new knowledge?"*
- Out of scope (must refuse): operational details such as Gemini API quotas.

### Metadata preserved per chunk
`source_id`, `title`, `url` / `source`, `retrieved_at`, `license`, `chunk_id`.

> **Design note — `MAX_CHARS_PER_DOC = 24000`.** The *Large language model* and
> *Transformer* articles are ~100k characters each; most of that tail is
> history, controversy and reception that adds noise to a *fundamentals*
> assistant. Capping each article at ~24k chars keeps the lead + core concept
> sections, holds the whole knowledge base to well under the Gemini free-tier
> embedding quota (100 requests/minute), and keeps ingestion fast. Remove the
> cap if you want full-article coverage.


In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

SOURCES = [
    {"source_id": "S1", "title": "Large language model",
     "url": "https://en.wikipedia.org/wiki/Large_language_model"},
    {"source_id": "S2", "title": "Transformer (deep learning architecture)",
     "url": "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)"},
    {"source_id": "S3", "title": "Retrieval-augmented generation",
     "url": "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"},
    {"source_id": "S4", "title": "Word embedding",
     "url": "https://en.wikipedia.org/wiki/Word_embedding"},
    {"source_id": "S5", "title": "Vector database",
     "url": "https://en.wikipedia.org/wiki/Vector_database"},
]

# Keep only the Wikipedia article body (drops navigation, sidebars, footer).
content_only = bs4.SoupStrainer("div", attrs={"class": "mw-body-content"})
retrieved_at = dt.date.today().isoformat()

raw_docs = []
for s in SOURCES:
    loader = WebBaseLoader(s["url"], bs_kwargs={"parse_only": content_only})
    for d in loader.load():
        d.page_content = d.page_content[:MAX_CHARS_PER_DOC]   # cap long articles
        d.metadata.update({
            "source_id": s["source_id"],
            "title": s["title"],
            "url": s["url"],
            "source": s["url"],
            "retrieved_at": retrieved_at,
            "license": "CC BY-SA 4.0",
        })
        raw_docs.append(d)

for d in raw_docs:
    print(f'{d.metadata["source_id"]}  {len(d.page_content):>7,} chars  {d.metadata["title"]}')

## 2. Build the knowledge base

### Chunking decisions

| Parameter | Value | Reason |
|-----------|-------|--------|
| splitter | `RecursiveCharacterTextSplitter` | splits on paragraph -> line -> sentence -> word, so chunks stay semantically coherent |
| `chunk_size` | **1200 chars** (~200-300 tokens) | large enough to hold a full definition or paragraph, small enough that one embedding stays "about one idea" |
| `chunk_overlap` | **200 chars** | carries a sentence or two across the boundary so an idea split between chunks is still retrievable |
| `top_k` | **4** | enough context for the model to synthesize an answer and cite 2-3 chunks, while keeping the prompt small and cheap on the free tier |

`chunk_id = "<source_id>:<n>"` is the citation handle used in answers, exactly
like `[rag:0]` in workshop notebook 03.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)

# stable, human-readable chunk ids
counters = {}
for c in chunks:
    sid = c.metadata["source_id"]
    counters[sid] = counters.get(sid, -1) + 1
    c.metadata["chunk_id"] = f"{sid}:{counters[sid]}"

print("total chunks:", len(chunks))
for sid, last in sorted(counters.items()):
    print(f"  {sid}: {last + 1} chunks")
print("\nexample chunk:\n", chunks[10].metadata["chunk_id"], "->",
      chunks[10].page_content[:160].replace("\n", " "), "...")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# gemini-embedding-001 returns 3072-d vectors by default. langchain-google-genai
# sends one API request per text, and the free tier allows 100 requests/minute,
# so ingestion below is throttled to stay under that limit.
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)


def add_in_batches(store, docs, batch_size=8, pause=7.0):
    """Ingest in small batches, ~65 embeddings/minute, to respect the Gemini
    free-tier limit (100 requests/min). On a 429 it waits and retries the same
    batch. Safe to re-run: if the collection already has data, ingestion is
    skipped, so the persisted DB is only built once."""
    if store.get(limit=1)["ids"]:
        print("collection already populated - skipping ingestion")
        return
    i = 0
    while i < len(docs):
        try:
            store.add_documents(docs[i:i + batch_size])
            i += batch_size
            print(f"  embedded {min(i, len(docs))}/{len(docs)}")
            time.sleep(pause)
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                print("  free-tier rate limit hit - waiting 60s, then retrying")
                time.sleep(60)
            else:
                raise


add_in_batches(vectorstore, chunks)
print("chunks in Chroma:", vectorstore._collection.count())

### Test retrieval before connecting the LLM

Inspect what the retriever returns on its own. If retrieval is wrong here, the
generated answer will be wrong too.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

for probe in [
    "What is the self-attention mechanism in a transformer?",
    "How does a vector database perform similarity search?",
]:
    print("Q:", probe)
    for d in retriever.invoke(probe):
        snippet = d.page_content[:150].replace("\n", " ")
        print(f'  [{d.metadata["chunk_id"]}] {d.metadata["title"]}: {snippet} ...')
    print()

## 3. Two-step RAG chain

`question -> retrieve (Chroma) -> build grounded prompt -> Gemini -> answer + sources`

The prompt forces the model to:
- use **only** the retrieved context,
- **cite chunk ids** in square brackets,
- **say what is missing** when the context is insufficient.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a study assistant for LLM and RAG fundamentals.
Answer the QUESTION using ONLY the CONTEXT below.

Rules:
- Do not use outside knowledge.
- If the context is insufficient, say explicitly what information is missing.
- Cite the chunk ids you used, in square brackets, e.g. [S2:4].
- Be concise (max ~6 sentences).

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""
)


def format_docs(docs):
    return "\n\n".join(
        f'[{d.metadata["chunk_id"]}] source: {d.metadata["title"]} ({d.metadata["url"]})\n'
        f'{d.page_content}'
        for d in docs
    )


# answer-only chain
rag_chain = (
    RunnableParallel(context=retriever | format_docs, question=RunnablePassthrough())
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# chain that also returns the retrieved documents (for citations / inspection)
rag_chain_with_sources = RunnableParallel(
    docs=retriever,
    question=RunnablePassthrough(),
).assign(
    answer=(
        RunnableParallel(
            context=lambda x: format_docs(x["docs"]),
            question=lambda x: x["question"],
        )
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
)


def ask_chain(question, show_chunks=True):
    out = rag_chain_with_sources.invoke(question)
    print("Q:", question)
    print("\nANSWER:\n", out["answer"])
    if show_chunks:
        print("\nRETRIEVED CHUNKS:")
        for d in out["docs"]:
            print(f'  [{d.metadata["chunk_id"]}] {d.metadata["title"]}')
    return out


_ = ask_chain("What problem does retrieval-augmented generation try to solve?")

## 4. RAG agent

Same retriever, exposed as a **tool**. A LangGraph ReAct agent decides on its
own whether to call the tool before answering. This mirrors the "tool-using
agent" pattern from workshop notebook `03` (section 11).


In [ ]:
from langchain_core.tools.retriever import create_retriever_tool
from langgraph.prebuilt import create_react_agent

retriever_tool = create_retriever_tool(
    retriever,
    name="search_llm_knowledge_base",
    description=(
        "Search the Wikipedia-based knowledge base about large language models, "
        "transformers, embeddings, retrieval-augmented generation and vector "
        "databases. Input: a natural-language question. Returns relevant chunks."
    ),
)

AGENT_SYSTEM = (
    "You are a study assistant. For any factual question, FIRST call "
    "search_llm_knowledge_base to gather evidence, then answer using ONLY that "
    "evidence and cite chunk ids in square brackets. If the evidence is "
    "insufficient, state exactly what is missing. Do not use outside knowledge."
)

agent = create_react_agent(llm, [retriever_tool])


def ask_agent(question, verbose=True):
    result = agent.invoke({"messages": [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": question},
    ]})
    msgs = result["messages"]
    tool_msgs = [m for m in msgs if getattr(m, "type", "") == "tool"]
    if verbose:
        print("Q:", question)
        print("tool calls:", len(tool_msgs))
        print("\nANSWER:\n", msgs[-1].content)
    return result


_ = ask_agent("What problem does retrieval-augmented generation try to solve?")

### 4b. Chain vs agent on the same question

In [ ]:
compare_q = "How does RAG help reduce hallucinations compared with a plain LLM?"

print("=" * 72)
print("TWO-STEP RAG CHAIN")
print("=" * 72)
chain_out = ask_chain(compare_q)

print("\n" + "=" * 72)
print("RAG AGENT")
print("=" * 72)
agent_out = ask_agent(compare_q)

n_tool_calls = len([m for m in agent_out["messages"] if getattr(m, "type", "") == "tool"])
print("\n--- quick comparison ---")
print("chain retrieved chunks :", [d.metadata["chunk_id"] for d in chain_out["docs"]])
print("agent tool calls       :", n_tool_calls)

### Comparison notes

Fill in / confirm after running the cell above.

| Aspect | Two-step RAG chain | RAG agent |
|--------|--------------------|-----------|
| Did it retrieve? | Yes, always (retrieval is hard-wired) | Yes, but only because the system prompt tells it to; it *chooses* to call the tool |
| Same sources used? | Compare the chunk ids above | Usually overlapping; the agent may issue a reworded query and get slightly different chunks |
| Similarly grounded? | Yes, prompt forces grounding + citations | Yes, same grounding instruction, but grounding depends on the agent actually calling the tool |
| Latency / cost | 1 LLM call | >= 2 LLM calls (decide -> tool -> answer) |
| Simplicity | Simpler, deterministic, easy to debug | More moving parts, useful when some questions need no retrieval or need multiple retrievals |

**Which fits this use case:** the **two-step chain**. Every question in this
domain needs the documents, so the agent's extra freedom (and extra LLM call)
buys nothing here. The agent pattern would pay off if the app also handled
chit-chat, multi-hop questions, or multiple tools.


## 5. Evaluation

Three questions:
1. **Answerable** from the documents.
2. **Partial / ambiguous** evidence.
3. **Not answerable** from the documents (should refuse).


In [ ]:
EVAL_QUESTIONS = [
    {"question": "What is the self-attention mechanism in the transformer architecture?",
     "type": "answerable"},
    {"question": "Is retrieval-augmented generation better than fine-tuning for adding new knowledge to an LLM?",
     "type": "partial / ambiguous"},
    {"question": "What are the exact requests-per-minute limits of the Gemini API free tier?",
     "type": "unanswerable from the documents"},
]

results = []
for item in EVAL_QUESTIONS:
    out = rag_chain_with_sources.invoke(item["question"])
    chunk_ids = [d.metadata["chunk_id"] for d in out["docs"]]
    titles = sorted({d.metadata["title"] for d in out["docs"]})
    print("#", item["type"].upper())
    print("Q:", item["question"])
    print("A:", out["answer"])
    print("retrieved:", chunk_ids)
    print("\nchunk previews:")
    for d in out["docs"]:
        print(f'  [{d.metadata["chunk_id"]}] {d.page_content[:180].strip().replace(chr(10), " ")} ...')
    print("-" * 72)
    results.append({
        "question": item["question"],
        "type": item["type"],
        "retrieved_sources": titles,
        "chunk_ids": chunk_ids,
        "answer": out["answer"],
    })

### Results table

Copy the produced values into this table for the README (and confirm the
*Grounded?* / *Observation* columns by reading the answers above).

| # | Question | Retrieved source | Result | Grounded? | Observation |
|---|----------|------------------|--------|-----------|-------------|
| 1 | What is the self-attention mechanism...? | S2 - Transformer | Correct definition of self-attention | Yes | Direct evidence in S2; answer cites S2 chunks |
| 2 | Is RAG better than fine-tuning...? | S3 - RAG (+ maybe S1) | Balanced but hedged answer | Partially | Articles mention the trade-off only briefly, so the model flags missing detail |
| 3 | Exact Gemini free-tier RPM limits? | S3 / S5 (low similarity) | Refusal: states the info is not in the documents | Yes (correct refusal) | Retrieval returns weakly-related chunks; prompt rule triggers the "what is missing" response |

### Required discussion

- **One case where retrieval worked well:** *(e.g. Q1 — "self-attention" maps
  cleanly onto S2 vocabulary, top chunks are the exact definition.)*
- **One failure / limitation:** *(e.g. Q2 — bag-of-concepts retrieval brings
  back the RAG article but not a crisp RAG-vs-fine-tuning comparison, because
  Wikipedia doesn't contain one; or a chunk boundary splits a definition.)*
- **One possible improvement:** *(e.g. add a RAG-vs-fine-tuning source; try
  `search_type="mmr"` for less redundant chunks; raise `top_k` to 6; or add a
  reranking step.)*

Replace the italic text with what you actually observe when you run the notebook.


## 6. Summary

- Documents -> `WebBaseLoader` -> `RecursiveCharacterTextSplitter` (1200/200)
  -> `gemini-embedding-001` -> **Chroma** (persisted, cosine).
- **Two-step chain**: retrieval is hard-wired; one Gemini call; deterministic;
  best fit for this use case.
- **RAG agent**: retrieval is a tool the model may call; >= 2 Gemini calls;
  useful only when some queries need no / multiple retrievals.
- Grounding is enforced by the prompt (context-only + citations + explicit
  "what is missing"), and verified by inspecting retrieved chunks next to the
  final answer.

This is the workshop's architecture (notebook 03) with each toy component
replaced by its production counterpart.
